In [16]:
import numpy as np 
import h5py
from egb_jax_eccentric import (
    EccentricBinaryParams,
    aet_from_xyz,
    eccentric_complex_strain,
    eccentric_links_jax,
    eccentric_xyz_jax,
    lisa_orbit,
    precompute_jax_link_geometry,
)
from egb_jax_eccentric.constants import PARSEC_M

In [17]:
import os
work_dir='/home/svu/e1498138/localgit/eGB-multi'
os.chdir(work_dir)


In [18]:
pwd

'/nfs/home/svu/e1498138/localgit/eGB-multi'

In [19]:
tdi_filepath=work_dir+'/data/tdi.h5'
with h5py.File(tdi_filepath, 'r') as f:
    # data in the h5 file 
    t = f["t"][:]
    x2 = f["X2"][:]
    y2 = f["Y2"][:]
    z2 = f["Z2"][:]

    # attributes
    dt = f.attrs["dt"]
    t0 = f.attrs["t0"]

In [20]:
t

array([1.0000000e+02, 1.0200000e+02, 1.0400000e+02, ..., 3.1557696e+07,
       3.1557698e+07, 3.1557700e+07], shape=(15778801,))

# Build LISA orbit/geometry

In [21]:
state = lisa_orbit(t)
geometry = precompute_jax_link_geometry(state)


# Verification Binary Injection

use HM Cnc (the first line in the LISA Verif binaries table). no eccentricity tho

In [22]:
# HM Cnc
period_s = 321.529129
pdot = -3.75e-11   
ra_deg=121.5957
dec_deg =15.4586
m1_sol=0.55
m2_sol=0.27
dist_pc = 7500.0  #pc
inc_deg = 38.0#deg

dont forget to convert to ecliptic

In [23]:
from astropy.coordinates import SkyCoord, BarycentricTrueEcliptic
import astropy.units as u

coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
ecliptic = coord.transform_to(BarycentricTrueEcliptic())
bG = ecliptic.lat.rad
lG = ecliptic.lon.rad

 Testing eccentricity values

In [24]:
source = EccentricBinaryParams(
    mean_motion=2.0 * np.pi / period_s,
    eccentricity=0.1, # NOTE: THIS IS TESTING OUT 0.1 
    m1_solar=m1_sol,
    m2_solar=m2_sol,
    beta=bG,
    lambda_=lG,
    psi=0.0,
    inclination=np.radians(inc_deg),
    phi0=0.0, #not sure 
    distance_m=dist_pc * PARSEC_M,
)

source

EccentricBinaryParams(mean_motion=0.01954157412338086, eccentricity=0.1, m1_solar=0.55, m2_solar=0.27, distance_m=2.3142581861185254e+20, inclination=np.float64(0.6632251157578453), beta=np.float64(-0.08210049881437637), lambda_=np.float64(2.102052384246663), psi=0.0, phi0=0.0, t0=0.0, fdot=0.0)

# Convert to XYZ

In [25]:
links = eccentric_links_jax(source, geometry, batch_size=1, physics_mode="1pn_periastron")
print({label: value.shape for label, value in links.items()})

xyz = eccentric_xyz_jax(
    state,
    source,
    geometry=geometry,
    batch_size=1,
    physics_mode="1pn_periastron",
    measurement_order=3,
    delay_order=3,
    generation=2 #NOTE: default in pytdi_bridge is 1
)

print({channel: value.shape for channel, value in xyz.items()})

{'21': (15778801,), '32': (15778801,), '13': (15778801,), '31': (15778801,), '23': (15778801,), '12': (15778801,)}


/scratch/e1498138/anaconda3/envs/egb-multi/lib/python3.11/site-packages/pytdi/dsp.py:72: ComplexWarning: Casting complex values to real discards the imaginary part
  return data.astype(float)


{'X': (15778801,), 'Y': (15778801,), 'Z': (15778801,)}


# Injection

In [26]:
N

15778801